#### **HOW TO USE THIS PROJECT TO GENERATE SYNTHETIC IMAGES ?**
1. Place this notebook in a separate folder
2. Choose the number of synthetic images you want to generate
3. Specify the prefix for your generated synthetic images (its usually the class name)
4. Specify the **"relative-path"** to the generator for that particular class
5. Run all cells
6. Your generated images should be stored at **"./generated_images/"**

In [2]:
num_images = 200
class_name = 'MoI'
path_to_generator = 'MoI_gen_2000_(12.43).h5'

#### **IMPORTING NECESSARY DEPENDENCIES**

In [3]:
import os
import cv2
import shutil
import numpy as np
import seaborn as sns
from tqdm import tqdm
from numpy import cov
from PIL import Image
from numpy import trace
import tensorflow as tf
from numpy import asarray
from tensorflow import keras
from scipy.linalg import sqrtm
from numpy import iscomplexobj
import matplotlib.pyplot as plt
from numpy.random import randint
from tensorflow.keras import layers
from skimage.transform import resize
from IPython.display import FileLink
from keras.datasets.mnist import load_data
from keras.applications.inception_v3 import InceptionV3
from skimage.metrics import structural_similarity as ssim
from keras.applications.inception_v3 import preprocess_input
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

#### **DEFINING IMPORTANT VARIABLES AND PATHS**

In [4]:
BATCH_SIZE = 64
noise_dim = 256
working_directory = './generated_images/'
os.makedirs(working_directory, exist_ok=True)
generator = tf.keras.models.load_model(path_to_generator)

#### **DEFINING IMPORTANT FUNCTIONS**

In [5]:
def clear_workspace():
    for filename in os.listdir(working_directory):
        file_path = os.path.join(working_directory, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print('Failed to delete %s. Reason: %s' % (file_path, e))

In [6]:
def download_synthetic_images(generator, number_of_images, class_name = 'None'):
    clear_workspace()
    # Making sure that no two images have the same name
    counter = 1
    # To check if folder is full
    folder_full = False
    # Folder path to store synthetic images. Folder name is the same as the class name which is inside the working directory
    #folder_path = working_directory + 'synthetic_' + class_name + '/'
    folder_path = working_directory
    images_folder = os.path.join(folder_path)
    # Create this folder if it doesn't exists, otherwise do nothing
    os.makedirs(images_folder, exist_ok=True)
    # Unless I get the desired number of good quality synthetic images in this folder, keep on generating images
    pbar = tqdm(total=number_of_images, position=0, leave=True)
    while len(os.listdir(images_folder)) < number_of_images:
        # Generate a batch of synthetic images
        random_latent_vectors = tf.random.normal(shape=(BATCH_SIZE, noise_dim))
        synthetic_images_batch = generator(random_latent_vectors)
        # Convert tensors to numpy arrays 
        synthetic_images_batch = synthetic_images_batch.numpy()
        # Convert the scale of images from (-1,1) to (0,255) and data type from float to integers 
        for image in synthetic_images_batch:
            image = (image * 127.5) + 127.5
            image = image.astype(np.uint8)
            # If it is a single channel image then save it as it is
            if image.shape[2] == 1:
                cv2.imwrite(images_folder + class_name + ' ({}).jpg'.format(counter), image)
                pbar.update(1)
                # Incrementing the counter so that no two images have the same name
                counter += 1
            # If it is a 3 channel image then first convert it to grayscale and then save it
            elif image.shape[2] == 3:
                cv2.imwrite(images_folder + class_name + ' ({}).jpg'.format(counter), image)
                pbar.update(1)
                # Incrementing the counter so that no two images have the same name
                counter += 1
            # If the folder now has the number of desired images, then come out of the loop
            if len(os.listdir(images_folder)) >= number_of_images:
                folder_full = True
                break
        if folder_full == True:
                break      
    return 'GENERATED AND SAVED ALL IMAGES'

#### **GENERATING AND SAVING SYNTHETIC IMAGES**

In [7]:
download_synthetic_images(generator, number_of_images = num_images, class_name = class_name)

100%|██████████| 200/200 [00:52<00:00,  3.84it/s]


'GENERATED AND SAVED ALL IMAGES'